# 美团商家评论：归因分析 + 评分预测模型

本 Notebook 汇总完整分析流程：

1. **数据加载**：训练集（stars 非空）/ 测试集（stars 为空）
2. **归因分析**：口味、环境、服务三维度 + 评论长度对好评的影响
3. **模型对比**：Count / TF-IDF / Word2Vec × RandomForest / XGBoost / NaiveBayes
4. **最终模型**：TF-IDF + 扩展特征 + GridSearch 调优 RandomForest
5. **报告输出**：归因报告、模型评估报告、预测结果与图表

> 数据文件：`data1.csv`  
> 产出目录：`output/`

## 0. 环境与依赖

In [4]:
%matplotlib inline

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from gensim.models import Word2Vec
from IPython.display import Markdown, display
from scipy.sparse import hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "PingFang SC", "Heiti SC"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path(".").resolve()
DATA_PATH = BASE_DIR / "data1.csv"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

DIM_SCORE = {"非常好": 5, "很好": 4, "好": 3, "一般": 2, "差": 1}
POSITIVE_LABELS = {"非常好", "很好", "好"}
GOOD_STAR_THRESHOLD = 4

ModuleNotFoundError: No module named 'gensim'

## 1. 数据加载

In [ ]:
def load_data():
    df = pd.read_csv(DATA_PATH)
    train_df = df[df["stars"].notna()].copy()
    test_df = df[df["stars"].isna()].copy()
    train_df["stars"] = train_df["stars"].astype(int)
    train_df["is_good"] = (train_df["stars"] >= GOOD_STAR_THRESHOLD).astype(int)
    return df, train_df, test_df


def map_dim_score(series):
    return series.map(DIM_SCORE)


df, train_df, test_df = load_data()
print(f"全量数据: {len(df):,} 条")
print(f"训练集 (stars 非空): {len(train_df):,} 条")
print(f"测试集 (stars 为空): {len(test_df):,} 条")
print(f"好评占比 (stars≥4): {train_df['is_good'].mean():.1%}")
train_df.head()

## 2. 归因分析

In [ ]:
def run_attribution_analysis(train_df):
    dims = {"口味": "kouwei", "环境": "huanjing", "服务": "fuwu"}
    good_df = train_df[train_df["is_good"] == 1]
    bad_df = train_df[train_df["is_good"] == 0]

    results = {}
    dim_positive_rates = {}

    for cn, col in dims.items():
        valid = good_df[good_df[col] != "无"]
        counts = valid[col].value_counts(normalize=True).sort_index()
        positive_rate = valid[col].isin(POSITIVE_LABELS).mean()
        dim_positive_rates[cn] = positive_rate
        dim_scores = map_dim_score(valid[col])
        results[cn] = {
            "distribution": counts.to_dict(),
            "positive_rate": float(positive_rate),
            "mean_score": float(dim_scores.mean()),
        }

    total_pos = sum(dim_positive_rates.values())
    dim_share = {k: v / total_pos for k, v in dim_positive_rates.items()}

    len_stats = {
        "good_mean": float(good_df["comment_len"].mean()),
        "bad_mean": float(bad_df["comment_len"].mean()),
        "good_median": float(good_df["comment_len"].median()),
        "bad_median": float(bad_df["comment_len"].median()),
        "overall_corr": float(train_df["comment_len"].corr(train_df["is_good"])),
    }

    bins = [0, 50, 100, 200, 500, 2000]
    labels = ["0-50", "51-100", "101-200", "201-500", "500+"]
    train_tmp = train_df.copy()
    train_tmp["len_bin"] = pd.cut(train_tmp["comment_len"], bins=bins, labels=labels, right=True)
    len_bin_good_rate = (
        train_tmp.groupby("len_bin", observed=True)["is_good"]
        .mean()
        .reindex(labels)
        .to_dict()
    )

    attribution = {
        "good_review_count": int(len(good_df)),
        "bad_review_count": int(len(bad_df)),
        "good_ratio": float(len(good_df) / len(train_df)),
        "dimension_analysis": results,
        "dimension_positive_share": dim_share,
        "comment_length": len_stats,
        "length_bin_good_rate": {str(k): float(v) for k, v in len_bin_good_rate.items()},
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    names = list(dim_positive_rates.keys())
    vals = list(dim_positive_rates.values())
    colors = sns.color_palette("Set2", 3)

    ax = axes[0, 0]
    ax.bar(names, vals, color=colors)
    ax.set_title("好评中各维度正向评价占比")
    ax.set_ylabel("正向评价比例")
    ax.set_ylim(0, 1)
    for i, v in enumerate(vals):
        ax.text(i, v + 0.01, f"{v:.1%}", ha="center")

    ax = axes[0, 1]
    ax.pie(dim_share.values(), labels=dim_share.keys(), autopct="%1.1f%%", colors=colors, startangle=90)
    ax.set_title("口味/服务/环境正向评价相对占比")

    ax = axes[1, 0]
    ax.hist(good_df["comment_len"], bins=50, alpha=0.6, label="好评", density=True)
    ax.hist(bad_df["comment_len"], bins=50, alpha=0.6, label="非好评", density=True)
    ax.set_xlabel("评论长度（字）")
    ax.set_ylabel("密度")
    ax.set_title("评论长度分布对比")
    ax.legend()

    ax = axes[1, 1]
    x = list(len_bin_good_rate.keys())
    y = list(len_bin_good_rate.values())
    ax.bar(x, y, color="steelblue")
    ax.set_xlabel("评论长度区间")
    ax.set_ylabel("好评率")
    ax.set_title("评论长度与好评率关系")
    ax.set_ylim(0, 1)
    for i, v in enumerate(y):
        ax.text(i, v + 0.02, f"{v:.1%}", ha="center")

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "attribution_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

    with open(OUTPUT_DIR / "attribution_results.json", "w", encoding="utf-8") as f:
        json.dump(attribution, f, ensure_ascii=False, indent=2)

    return attribution


attribution = run_attribution_analysis(train_df)

In [ ]:
def generate_attribution_report(attribution):
    da = attribution["dimension_analysis"]
    ds = attribution["dimension_positive_share"]
    cl = attribution["comment_length"]
    lb = attribution["length_bin_good_rate"]

    report = f"""# 影响用户好评的关键因素归因报告

## 一、数据概况

| 指标 | 数值 |
|------|------|
| 好评样本数（stars≥4） | {attribution['good_review_count']:,} |
| 非好评样本数（stars≤3） | {attribution['bad_review_count']:,} |
| 好评占比 | {attribution['good_ratio']:.1%} |

> 好评定义：综合评分 stars ≥ 4 分（4分与5分）。

---

## 二、口味、服务、环境在好评中的占比分析

| 维度 | 正向评价占比 | 平均得分（1-5） |
|------|-------------|----------------|
| 口味 | {da['口味']['positive_rate']:.1%} | {da['口味']['mean_score']:.2f} |
| 环境 | {da['环境']['positive_rate']:.1%} | {da['环境']['mean_score']:.2f} |
| 服务 | {da['服务']['positive_rate']:.1%} | {da['服务']['mean_score']:.2f} |

### 三维度正向评价相对占比

| 维度 | 相对占比 |
|------|---------|
| 口味 | {ds['口味']:.1%} |
| 环境 | {ds['环境']:.1%} |
| 服务 | {ds['服务']:.1%} |

### 归因结论（维度）

1. **口味是核心驱动因素**：正向占比 {da['口味']['positive_rate']:.1%}，相对贡献 {ds['口味']:.1%}。
2. **环境**：正向占比 {da['环境']['positive_rate']:.1%}。
3. **服务相对短板**：正向占比 {da['服务']['positive_rate']:.1%}，为三维度最低。

---

## 三、评论长短与好评的归因分析

| 指标 | 好评 | 非好评 |
|------|------|--------|
| 平均评论长度 | {cl['good_mean']:.1f} 字 | {cl['bad_mean']:.1f} 字 |
| 中位评论长度 | {cl['good_median']:.1f} 字 | {cl['bad_median']:.1f} 字 |

- Pearson 相关系数：**{cl['overall_corr']:.4f}**

### 评论长度分箱 vs 好评率

| 长度区间（字） | 好评率 |
|---------------|--------|
"""
    for k, v in lb.items():
        report += f"| {k} | {v:.1%} |\n"

    report += f"""
### 归因结论（评论长度）

1. 评论长度与好评呈弱负相关（r = {cl['overall_corr']:.4f}）。
2. 0-50 字区间好评率最高（{lb.get('0-50', 0):.1%}），500+ 字最低（{lb.get('500+', 0):.1%}）。
3. 101-200 字为次优区间（{lb.get('101-200', 0):.1%}）。

---

## 四、综合归因总结

| 因素 | 影响方向 | 重要程度 |
|------|---------|---------|
| 口味 | 正向驱动 | ★★★★★ |
| 环境 | 正向驱动 | ★★★★ |
| 服务 | 相对弱项 | ★★★ |
| 评论长度 | 短评略关联好评 | ★★ |

**核心建议**：优先保障菜品口味稳定性；改善等位与服务态度可显著提升综合评分。
"""
    (OUTPUT_DIR / "影响用户好评的关键因素归因报告.md").write_text(report, encoding="utf-8")
    return report


attribution_report = generate_attribution_report(attribution)
display(Markdown(attribution_report))

## 3. 模型训练与对比

In [ ]:
def prepare_text(text_series):
    texts = []
    for t in text_series.fillna(""):
        t = str(t).strip()
        texts.append(t if t else "")
    return texts


def build_word2vec_features(texts, vector_size=100):
    sentences = [t.split() if t else [] for t in texts]
    w2v = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4,
        seed=42,
        epochs=10,
    )

    def doc_vec(tokens):
        vecs = [w2v.wv[w] for w in tokens if w in w2v.wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)

    return np.array([doc_vec(s) for s in sentences]), w2v


def train_and_evaluate_models(train_df, test_df):
    texts = prepare_text(train_df["cus_comment"])
    X_text_train, X_text_test = texts, prepare_text(test_df["cus_comment"])
    y = train_df["stars"].values

    indices = np.arange(len(train_df))
    idx_tr, idx_val = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
    y_tr, y_val = y[idx_tr], y[idx_val]

    count_vec = CountVectorizer(max_features=5000)
    X_count_tr = count_vec.fit_transform([X_text_train[i] for i in idx_tr])
    X_count_val = count_vec.transform([X_text_train[i] for i in idx_val])
    X_count_test = count_vec.transform(X_text_test)

    tfidf_vec = TfidfVectorizer(max_features=5000)
    X_tfidf_tr = tfidf_vec.fit_transform([X_text_train[i] for i in idx_tr])
    X_tfidf_val = tfidf_vec.transform([X_text_train[i] for i in idx_val])
    X_tfidf_test = tfidf_vec.transform(X_text_test)

    w2v_tr, w2v_model = build_word2vec_features([X_text_train[i] for i in idx_tr])
    w2v_val = np.array([
        np.mean([w2v_model.wv[w] for w in X_text_train[i].split() if w in w2v_model.wv], axis=0)
        if any(w in w2v_model.wv for w in X_text_train[i].split())
        else np.zeros(w2v_model.vector_size)
        for i in idx_val
    ])
    w2v_test = np.array([
        np.mean([w2v_model.wv[w] for w in t.split() if w in w2v_model.wv], axis=0)
        if any(w in w2v_model.wv for w in t.split())
        else np.zeros(w2v_model.vector_size)
        for t in X_text_test
    ])

    feature_configs = {
        "Count-onehot": (X_count_tr, X_count_val, X_count_test, count_vec),
        "TF-IDF": (X_tfidf_tr, X_tfidf_val, X_tfidf_test, tfidf_vec),
        "Word2Vec": (w2v_tr, w2v_val, w2v_test, w2v_model),
    }

    classifiers = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, use_label_encoder=False, eval_metric="mlogloss", verbosity=0,
        ),
        "NaiveBayes": MultinomialNB(alpha=0.1),
    }

    comparison_rows = []
    for feat_name, (X_tr_f, X_val_f, X_test_f, vec_obj) in feature_configs.items():
        for clf_name, clf_template in classifiers.items():
            if clf_name == "NaiveBayes" and feat_name == "Word2Vec":
                continue
            clf = clf_template.__class__(**clf_template.get_params())
            if clf_name == "XGBoost":
                le = LabelEncoder()
                le.fit([1, 2, 3, 4, 5])
                clf.fit(X_tr_f, le.transform(y_tr))
                y_pred = le.inverse_transform(clf.predict(X_val_f).astype(int))
            else:
                clf.fit(X_tr_f, y_tr)
                y_pred = clf.predict(X_val_f)
            comparison_rows.append({
                "特征": feat_name,
                "分类器": clf_name,
                "准确率": accuracy_score(y_val, y_pred),
                "F1_macro": f1_score(y_val, y_pred, average="macro"),
                "F1_weighted": f1_score(y_val, y_pred, average="weighted"),
            })

    comparison_df = pd.DataFrame(comparison_rows).sort_values("F1_macro", ascending=False)
    comparison_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False, encoding="utf-8-sig")

    dim_cols = ["kouwei", "huanjing", "fuwu"]
    dim_train = pd.get_dummies(train_df[dim_cols], prefix=dim_cols)
    dim_test = pd.get_dummies(test_df[dim_cols], prefix=dim_cols).reindex(columns=dim_train.columns, fill_value=0)

    return {
        "comparison_df": comparison_df,
        "X_text_train": X_text_train,
        "X_text_test": X_text_test,
        "y": y,
        "extra_cols": ["comment_len", "year", "month", "weekday", "hour"],
        "dim_train": dim_train,
        "dim_test": dim_test,
        "train_df": train_df,
        "test_df": test_df,
    }


print("模型训练与对比中（约需数分钟）...")
ctx = train_and_evaluate_models(train_df, test_df)
comparison_df = ctx["comparison_df"]
comparison_df

## 4. 最终模型调优与预测

In [ ]:
def tune_and_build_final_model(ctx):
    train_df = ctx["train_df"]
    test_df = ctx["test_df"]
    X_text_train = ctx["X_text_train"]
    X_text_test = ctx["X_text_test"]
    y = ctx["y"]
    extra_cols = ctx["extra_cols"]
    dim_train = ctx["dim_train"]
    dim_test = ctx["dim_test"]

    tfidf_vec = TfidfVectorizer(max_features=8000, ngram_range=(1, 2))
    X_tfidf = tfidf_vec.fit_transform(X_text_train)
    X_tfidf_test = tfidf_vec.transform(X_text_test)

    X_full = hstack([X_tfidf, train_df[extra_cols].values, dim_train.values])
    X_full_test = hstack([X_tfidf_test, test_df[extra_cols].values, dim_test.values])

    X_tr, X_val, y_tr, y_val = train_test_split(X_full, y, test_size=0.2, random_state=42, stratify=y)

    grid = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators": [100, 200], "max_depth": [None, 20, 40], "min_samples_split": [2, 5]},
        cv=3, scoring="f1_macro", n_jobs=-1, verbose=0,
    )
    grid.fit(X_tr, y_tr)
    best_model = grid.best_estimator_
    y_val_pred = best_model.predict(X_val)

    val_metrics = {
        "accuracy": float(accuracy_score(y_val, y_val_pred)),
        "f1_macro": float(f1_score(y_val, y_val_pred, average="macro")),
        "f1_weighted": float(f1_score(y_val, y_val_pred, average="weighted")),
        "best_params": grid.best_params_,
    }

    best_model.fit(X_full, y)
    test_pred = best_model.predict(X_full_test)

    joblib.dump({
        "model": best_model,
        "tfidf_vectorizer": tfidf_vec,
        "extra_cols": extra_cols,
        "dim_columns": list(dim_train.columns),
        "best_params": grid.best_params_,
    }, OUTPUT_DIR / "final_model.pkl")

    test_df_out = test_df.copy()
    test_df_out["predicted_stars"] = test_pred
    test_df_out.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")

    report = classification_report(y_val, y_val_pred, digits=4)
    cm = confusion_matrix(y_val, y_val_pred)

    with open(OUTPUT_DIR / "final_model_eval.json", "w", encoding="utf-8") as f:
        json.dump({"validation_metrics": val_metrics, "classification_report": report, "confusion_matrix": cm.tolist()}, f, ensure_ascii=False, indent=2)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("预测评分")
    ax.set_ylabel("真实评分")
    ax.set_title("最终模型验证集混淆矩阵")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    return val_metrics, report, test_pred


print("最终模型调优中（GridSearch，约需数分钟）...")
val_metrics, class_report, test_pred = tune_and_build_final_model(ctx)
print(f"准确率: {val_metrics['accuracy']:.4f}")
print(f"F1_macro: {val_metrics['f1_macro']:.4f}")
print(f"最优参数: {val_metrics['best_params']}")

In [ ]:
def generate_model_report(comparison_df, val_metrics, class_report):
    try:
        comp_table = comparison_df.to_markdown(index=False, floatfmt=".4f")
    except ImportError:
        comp_table = comparison_df.to_string(index=False)

    report = f"""# 评分预测模型性能评估报告

## 一、任务说明

- **目标**：基于评论文本及辅助特征，预测用户综合评分（stars，1-5 分）
- **训练集**：stars 非空，共 {len(train_df):,} 条
- **测试集**：stars 为空，共 {len(test_df):,} 条
- **验证方式**：训练集 80/20 分层抽样

---

## 二、文本向量化方法

| 方法 | 说明 |
|------|------|
| Count-onehot | 词袋模型，最大 5000 特征 |
| TF-IDF | 词频-逆文档频率，最大 5000 特征 |
| Word2Vec | 100 维词向量，文档向量取词向量均值 |

---

## 三、分类器对比结果

{comp_table}

**最优基线组合**：{comparison_df.iloc[0]['特征']} + {comparison_df.iloc[0]['分类器']}（F1_macro = {comparison_df.iloc[0]['F1_macro']:.4f}）

---

## 四、最终优化模型

### 验证集性能

| 指标 | 数值 |
|------|------|
| 准确率 | {val_metrics['accuracy']:.4f} |
| F1 (macro) | {val_metrics['f1_macro']:.4f} |
| F1 (weighted) | {val_metrics['f1_weighted']:.4f} |

**最优参数**：
```json
{json.dumps(val_metrics['best_params'], ensure_ascii=False, indent=2)}
```

### 分类报告

```
{class_report}
```

---

## 五、结论与建议

1. **TF-IDF 特征**在稀疏文本分类任务中整体优于 Count 与 Word2Vec。
2. **RandomForest** 在引入维度与上下文特征后表现显著提升。
3. 测试集预测已保存至 `output/test_predictions.csv`。
"""
    (OUTPUT_DIR / "模型性能评估报告.md").write_text(report, encoding="utf-8")
    return report


model_report = generate_model_report(comparison_df, val_metrics, class_report)
display(Markdown(model_report))

## 5. 测试集预测预览

In [ ]:
pred_df = pd.read_csv(OUTPUT_DIR / "test_predictions.csv")
print(f"测试集预测完成，共 {len(pred_df):,} 条")
print("\n预测评分分布：")
print(pred_df["predicted_stars"].value_counts().sort_index())
pred_df[["cus_comment", "kouwei", "huanjing", "fuwu", "comment_len", "predicted_stars"]].head(10)

## 6. 产出文件清单

| 文件 | 说明 |
|------|------|
| `output/attribution_analysis.png` | 归因分析四宫格图 |
| `output/attribution_results.json` | 归因分析原始数据 |
| `output/影响用户好评的关键因素归因报告.md` | 归因报告 |
| `output/model_comparison.csv` | 模型对比表 |
| `output/final_model.pkl` | 最终模型 |
| `output/final_model_eval.json` | 模型评估指标 |
| `output/confusion_matrix.png` | 混淆矩阵图 |
| `output/test_predictions.csv` | 测试集预测结果 |
| `output/模型性能评估报告.md` | 模型评估报告 |

> 如需导出 PDF，可运行 `export_reports_pdf.py`。